# Fully Offline Audio Summarizer & Diarization Pipeline

This notebook transcribes audio files, attributes speech to specific speakers, maps generic speaker IDs to real names, and generates an executive summary document using a local LLM via Map-Reduce summarization.

### Prerequisites:
1. Ran initial caching script online (`whisper`, `PyAnnote`, `Qwen2.5-7B-Instruct`).
2. Set up WSL2 + ROCm PyTorch environment.

In [ ]:
import os
import gc
import re
import torch
import whisper
from pyannote.audio import Pipeline
from docx import Document
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------------------------------------------------------------------------
# Global Configurations
# ---------------------------------------------------------------------------
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
WHISPER_MODEL_SIZE = "large-v3"
PYANNOTE_PIPELINE_ID = "pyannote/speaker-diarization-3.1"
OUTPUT_DOC_TYPE = "pdf"  # Options: 'docx' or 'pdf'

# Force Hugging Face ecosystem to run 100% offline
os.environ["HF_HUB_OFFLINE"] = "1"

def free_vram():
    """Forces garbage collection and clears PyTorch CUDA/ROCm memory cache."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("[VRAM] Cache cleared.")

print("Environment initialized and offline mode enforced.")

In [ ]:
def diarize_audio(audio_path: str):
    """Stage 1: Detect speaker intervals using PyAnnote."""
    print(f"\n[1/5] Loading PyAnnote Diarization pipeline...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    pipeline = Pipeline.from_pretrained(PYANNOTE_PIPELINE_ID)
    pipeline.to(device)
    
    print(f"[1/5] Identifying speakers in '{audio_path}'...")
    diarization_result = pipeline(audio_path)
    
    del pipeline
    free_vram()
    return diarization_result

In [ ]:
def transcribe_and_align(audio_path: str, diarization_result) -> str:
    """Stage 2: Transcribe speech with Whisper and align text segments with speaker intervals."""
    print(f"\n[2/5] Loading Whisper ({WHISPER_MODEL_SIZE})...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)
    
    print("[2/5] Transcribing audio and extracting word timestamps...")
    result = model.transcribe(audio_path, verbose=False)
    whisper_segments = result["segments"]
    
    del model
    free_vram()
    
    print("[2/5] Aligning speaker IDs to text segments...")
    speaker_transcript = []
    
    for segment in whisper_segments:
        seg_start = segment['start']
        seg_end = segment['end']
        seg_text = segment['text'].strip()
        
        best_speaker = "Unknown Speaker"
        max_overlap = 0.0
        
        for turn, _, speaker in diarization_result.itertracks(yield_label=True):
            overlap = max(0, min(seg_end, turn.end) - max(seg_start, turn.start))
            if overlap > max_overlap:
                max_overlap = overlap
                best_speaker = speaker
                
        speaker_transcript.append(f"[{best_speaker}] {seg_text}")
        
    return "\n".join(speaker_transcript)

In [ ]:
def map_speakers_interactively(transcript: str) -> str:
    """Stage 3: Interactive rename of generic speaker tags (e.g. SPEAKER_00 -> Sarah)."""
    unique_speakers = sorted(set(re.findall(r'\[(SPEAKER_\d{2,})\]', transcript)))
    
    if not unique_speakers:
        print("[3/5] No standard speaker tags found to map.")
        return transcript
        
    print("\n[3/5] Interactive Speaker Mapping")
    print("-" * 45)
    print("Transcript preview:")
    print(transcript[:350])
    print("-" * 45)
    print("Enter real names for detected speakers (press Enter to keep default):")
    
    mapping_dict = {}
    for speaker in unique_speakers:
        real_name = input(f" -> Rename [{speaker}]: ").strip()
        if real_name:
            mapping_dict[f"[{speaker}]"] = f"[{real_name}]"
            
    mapped_transcript = transcript
    for old_tag, new_tag in mapping_dict.items():
        mapped_transcript = mapped_transcript.replace(old_tag, new_tag)
        
    print("[3/5] Speaker mapping complete.")
    return mapped_transcript

In [ ]:
def chunk_text(text: str, max_chars: int = 6000, overlap: int = 500) -> list:
    """Splits long transcripts into overlapping chunks to fit within GPU KV-cache limits."""
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = min(start + max_chars, text_length)
        if end < text_length:
            end = text.rfind(' ', start, end)
            if end == -1:
                end = start + max_chars
                
        chunks.append(text[start:end])
        start = end - overlap
        
    return chunks

def generate_summary(transcript: str) -> str:
    """Stage 4: Map-Reduce LLM Summarization using HuggingFace."""
    print(f"\n[4/5] Loading LLM ({LLM_MODEL_ID}) into VRAM...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=True
    )
    
    # 1. MAP PHASE
    chunks = chunk_text(transcript)
    chunk_summaries = []
    
    map_prompt_template = (
        "You are an assistant analyzing a portion of a meeting transcript. "
        "Extract key points, decisions, financial/technical details, and action items from this segment.\n\n"
        "Segment:\n{text}"
    )
    
    print(f"[4/5] Map Phase: Processing {len(chunks)} transcript chunk(s)...")
    for i, chunk in enumerate(chunks):
        print(f"      Summarizing chunk {i+1}/{len(chunks)}...")
        messages = [
            {"role": "system", "content": "You are a precise executive assistant."},
            {"role": "user", "content": map_prompt_template.format(text=chunk)}
        ]
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
        
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=500,
                temperature=0.3,
                pad_token_id=tokenizer.eos_token_id
            )
            
        generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        chunk_summaries.append(tokenizer.decode(generated_tokens, skip_special_tokens=True))
        
        del inputs
        del output_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 2. REDUCE PHASE
    print("[4/5] Reduce Phase: Synthesizing master summary...")
    combined_summaries = "\n\n".join([f"Part {i+1}:\n{s}" for i, s in enumerate(chunk_summaries)])
    
    reduce_prompt = (
        "You are an executive assistant. Synthesize these partial meeting notes into a cohesive master document in Markdown.\n\n"
        "Structure:\n"
        "1. Executive Summary\n"
        "2. Key Discussions & Decisions\n"
        "3. Financial, Technical & Strategic Highlights\n"
        "4. Action Items & Next Steps (Attributed to specific speakers)\n\n"
        f"Partial Notes:\n{combined_summaries}"
    )
    
    messages = [
        {"role": "system", "content": "You synthesize meeting transcripts into clear executive summaries."},
        {"role": "user", "content": reduce_prompt}
    ]
    
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id
        )
        
    generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    final_summary = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    del model
    del tokenizer
    free_vram()
    
    return final_summary

In [ ]:
def export_docx(text: str, output_path: str):
    """Stage 5: Export Markdown summary to Word document."""
    doc = Document()
    doc.add_heading("Meeting Summary Document", 0)
    
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("# "):
            doc.add_heading(line[2:], level=1)
        elif line.startswith("## "):
            doc.add_heading(line[3:], level=2)
        elif line.startswith("### "):
            doc.add_heading(line[4:], level=3)
        elif line.startswith("- ") or line.startswith("* "):
            doc.add_paragraph(line[2:], style='List Bullet')
        elif line:
            doc.add_paragraph(line)
            
    doc.save(output_path)
    print(f"[5/5] Saved Word document: {output_path}")

def export_pdf(text: str, output_path: str):
    """Stage 5: Export Markdown summary to PDF document."""
    doc = SimpleDocTemplate(output_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("Meeting Summary Document", styles['Title']))
    story.append(Spacer(1, 12))

    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("#"):
            clean_line = line.lstrip("#").strip()
            story.append(Spacer(1, 10))
            story.append(Paragraph(clean_line, styles['Heading2']))
            story.append(Spacer(1, 6))
        elif line:
            story.append(Paragraph(line, styles['BodyText']))
            story.append(Spacer(1, 4))

    doc.build(story)
    print(f"[5/5] Saved PDF document: {output_path}")

In [ ]:
# ===========================================================================
# Pipeline Execution Cell
# ===========================================================================

# Specify path to your target audio file
AUDIO_FILE_PATH = "meeting.mp3" 

if not os.path.exists(AUDIO_FILE_PATH):
    print(f"Error: Target audio file '{AUDIO_FILE_PATH}' does not exist.")
else:
    print("=== Pipeline Initiated ===")
    
    # Step 1: Diarize
    diarization_data = diarize_audio(AUDIO_FILE_PATH)
    
    # Step 2: Transcribe & Align
    raw_transcript = transcribe_and_align(AUDIO_FILE_PATH, diarization_data)
    
    # Step 3: Interactive Speaker Mapping
    named_transcript = map_speakers_interactively(raw_transcript)
    
    # Step 4: Summarize
    summary_markdown = generate_summary(named_transcript)
    
    # Step 5: Document Export
    base_filename = os.path.splitext(AUDIO_FILE_PATH)[0]
    if OUTPUT_DOC_TYPE == "docx":
        export_docx(summary_markdown, f"{base_filename}_Summary.docx")
    else:
        export_pdf(summary_markdown, f"{base_filename}_Summary.pdf")
        
    print("\n=== Pipeline Complete ===")